In [ ]:
from fastapi import FastAPI, HTTPException
import httpx
from bs4 import BeautifulSoup

In [ ]:
# Cabeçalhos globais para simular um navegador real em todas as requisições
HEADERS = {
    "accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "accept-language": "pt-BR,pt;q=0.9,en-US;q=0.8",
    "user-agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
}

In [ ]:
url = "https://www.carrosnaweb.com.br/"
    
# Passando os cabeçalhos atualizados para o cliente HTTP
async with httpx.AsyncClient(timeout=10.0) as client:
    response = await client.get(url, headers=HEADERS)
    
if response.status_code != 200:
    raise HTTPException(
        status_code=502, 
        detail="Falha ao acessar o diretório de veículos"
    )

In [ ]:
soup = BeautifulSoup(response.text, "html.parser")
select_modelo = soup.find("select", id="valfim")

In [ ]:
import re 
# Supondo que 'soup' já tenha sido instanciado pelo BeautifulSoup

fonts = soup.find_all('font')
total_paginas = None


for font in fonts:
    texto = font.get_text().strip()
    
    # Busca por "P" + qualquer coisa + "gina", números, "de" e captura o último número
    match = re.search(r'P.*?gina\s+\d+\s+de\s+(\d+)', texto, re.IGNORECASE)
    
    if match:
        total_paginas = int(match.group(1)) # O group(1) retorna o que está entre parênteses na Regex
        break

urls_para_raspar = [url]
for i in range(total_paginas):
    
    url_base = f'{url}&curpage={i}'
    async with httpx.AsyncClient(timeout=15.0) as client:
            response = await client.get(url_base, headers=self.headers)
            if response.status_code == 200:
                urls_para_raspar.append(url_base)
                
            if response.status_code != 200:
                print(f"Aviso: Falha ao acessar {url_base}")
                
    


In [ ]:
fonts

In [ ]:
import re

max_pagina = 1

urls_para_raspar = []

for a_tag in soup.find_all("a", href=True):
    href = a_tag["href"]
    if "curpage=" in href:
        # 2. Usa Regex para extrair apenas o número da página. Ex: curpage=5 -> pega o 5
        match = re.search(r'curpage=(\d+)', href)
        if match:
            num_pagina = int(match.group(1))
            # Mantém sempre o maior número encontrado
            if num_pagina > max_pagina:
                max_pagina = num_pagina
        
     # 3. Se encontrou mais de 1 página, gera o restante das URLs adicionando o parâmetro
    if max_pagina > 1:
        for pagina in range(2, max_pagina + 1):
            # Como a sua url_base não tem o curpage, adicionamos no final
            nova_url = f"{url}&curpage={pagina}"
            urls_para_raspar.append(nova_url)    

print(urls_para_raspar)               
                


In [ ]:
from fastapi import HTTPException
from bs4 import BeautifulSoup
import httpx

# Cabeçalhos definidos globalmente (como você já tinha feito antes)
HEADERS = {
    "accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "accept-language": "pt-BR,pt;q=0.9,en-US;q=0.8",
    "user-agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
}

# A função precisa ser 'async def' para usar o 'await' do httpx
async def extrair_versoes(url: str) -> list[dict]:
    
    # Passando os cabeçalhos atualizados para o cliente HTTP
    async with httpx.AsyncClient(timeout=10.0) as client:
        response = await client.get(url, headers=HEADERS)
        
    if response.status_code != 200:
        raise HTTPException(
            status_code=502, 
            detail="Falha ao acessar o diretório de veículos"
        )
    
    soup = BeautifulSoup(response.text, "html.parser")
    versoes_encontradas = []

    # 1. Encontra todas as células de tabela (td) alinhadas à esquerda (onde ficam os textos)
    celulas_info = soup.find_all("td", align="left")

    for td in celulas_info:
        # 2. Busca a tag de link (<a>) dentro dessa célula
        link_tag = td.find("a", href=True)
        
        # Garante que é o link correto da ficha técnica
        if link_tag and "fichadetalhe.asp" in link_tag["href"]:
            href = link_tag["href"]
            codigo = href.split("codigo=")[-1]
            url_completa = f"https://www.carrosnaweb.com.br/{href}"
            
            # 3. O stripped_strings pega os textos e ignora o HTML
            textos = list(link_tag.stripped_strings)
            
            if len(textos) >= 3:
                # O \xa0 é o código para "&nbsp;" (espaço inquebrável) no HTML
                versao_limpa = textos[1].replace('\xa0', ' ')
                
                versoes_encontradas.append({
                    "id_veiculo": codigo,
                    "fabricante": textos[0],
                    "versao": versao_limpa,
                    "ano": textos[2],
                    "url_ficha": url_completa
                })
                
    return versoes_encontradas

In [ ]:
url = 'https://www.carrosnaweb.com.br/catalogo.asp?fabricante=Ford&porte=0&varnome=Ranger'

dados_extraidos = await extrair_versoes(url)

In [ ]:
dados_extraidos

In [ ]:
from fastapi import FastAPI, HTTPException
# Importe a classe que criamos. Ajuste o caminho conforme o local do seu arquivo principal
from src.utils.CarrosWeb import CarrosWeb

carrosWeb = CarrosWeb()

carros = await carrosWeb.obter_todas_urls_paginacao("https://www.carrosnaweb.com.br/catalogo.asp?fabricante=Ford&modelo=Ranger")

In [3]:
carros = await carrosWeb.obter_todas_urls_paginacao("https://www.carrosnaweb.com.br/catalogo.asp?fabricante=Ford&modelo=Ranger")

[!] Erro 200 ou bloqueio. Tentativa 1/3.
[i] Rotacionando impersonate para: edge101
[!] Erro 200 ou bloqueio. Tentativa 2/3.
[i] Rotacionando impersonate para: chrome110
[!] Erro 200 ou bloqueio. Tentativa 3/3.


In [ ]:
carros

In [ ]:
urls_para_raspar = [url_base]

        # Não precisa mais visitar a home manualmente aqui —
        # o _get() já garante isso via _bootstrap_sessao() na primeira chamada
        soup = await self._get(url_base)

        if not soup:
            return urls_para_raspar

        fonts = soup.find_all('font')
        total_paginas = 1

        for font in fonts:
            texto = font.get_text().strip()
            match = re.search(r'P.*?gina\s+\d+\s+de\s+(\d+)', texto, re.IGNORECASE)
            if match:
                total_paginas = int(match.group(1))
                break

        if total_paginas > 1:
            for i in range(2, total_paginas + 1):
                nova_url = f"{url_base}&curpage={i}"

                # Testamos rapidamente se a página 2, 3, etc. existe
                # retries=1 faz o processo ser mais ágil
                soup_teste = await self._get(nova_url, retries=1)

                if soup_teste:
                    urls_para_raspar.append(nova_url)

        return urls_para_raspar